# Generative AI API Playground

A hands-on notebook for experimenting with **Generative AI APIs** and simple multimodal applications in Python.

Covered topics include:

- Chat completions with OpenAI-compatible APIs
- Cohere chat API
- AvalAI / OpenAI-compatible endpoints
- Text-to-Speech (TTS) and Speech-to-Text (STT)
- Image generation and image understanding
- OpenRouter / multimodal model calls
- LangChain integration
- Interactive Gradio applications for chat, vision, and speech

> **Learning context:** This notebook is a cleaned and organized version of hands-on exercises completed while studying Generative AI APIs. External models, APIs, and libraries remain credited to their original providers.

## API Key Configuration

This notebook reads API keys from environment variables instead of hard-coding credentials.

Set the variables below in your environment or Colab session before running API-dependent cells:

- `OPENAI_API_KEY`
- `COHERE_API_KEY`
- `AVALAI_API_KEY`
- `OPENROUTER_API_KEY`

In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
AVALAI_API_KEY = os.getenv("AVALAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

## API for AI

# Part 1 — OpenAI API

In [ ]:
!pip install -q openai

In [ ]:
messages = [
    {"role": "system", "content": "You are a kind helpful PERSIAN assistant and response in Farsi and you can give me SQL Query"},
]

In [ ]:
from openai import OpenAI

client = OpenAI(
        api_key=OPENAI_API_KEY

)

stream = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ],
    model="gpt-3.5-turbo",
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)
messages = []  # Initialize messages list

while True:
    message = input("User: ")
    if message.lower() in ["exit", "quit"]:  # Allow user to exit
        break

    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(  # Updated function call
        model="gpt-3.5-turbo",
        messages=messages
    )

    reply = response.choices[0].message.content
    print(f"ChatGPT: {reply}")

    messages.append({"role": "assistant", "content": reply})  # Save bot's response

# Part 2 — Cohere API

In [ ]:
!pip install cohere

In [ ]:
import cohere

co = cohere.ClientV2(api_key=COHERE_API_KEY)

res = co.chat(
    model="command-r-plus-08-2024",
    messages=[
        {
            "role": "user",
            "content": "برای یک پروژه دستیار هوش مصنوعی، چه قابلیت هایی را پیشنهاد می کنی؟",
        }
    ],
)

print(res.message.content[0].text)
# "The Ultimate Guide to API Design: Best Practices for Building Robust and Scalable APIs"

In [ ]:
import cohere

co = cohere.ClientV2(api_key=COHERE_API_KEY)

system_message = "You respond concisely, in about 20 words or less"

res = co.chat(
    model="command-r-plus-08-2024",
    messages=[
        {"role": "system", "content": system_message},
        {
            "role": "user",
            "content": "من دندان درد دارم",
        }
    ],
)

# "AI: The Generative Age"
print(res.message.content[0].text)

# Part 3 — AvalAI / OpenAI-Compatible API

In [ ]:
import requests

url = "https://api.avalai.ir/v1/models"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AVALAI_API_KEY}"
}
response = requests.get(url, headers=headers)
print(response.json())

In [ ]:
!pip install -U openai
!pip install -U langchain
!pip install -U langchain_openai

In [ ]:
from openai import OpenAI # pip install -U openai

client = OpenAI(
base_url="https://api.avalai.ir/v1",
api_key=AVALAI_API_KEY
)

speech_file_path = "./speech.mp3"

response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input="این یک نمونه تبدیل متن فارسی به گفتار با استفاده از API هوش مصنوعی است",
)

response.stream_to_file(speech_file_path)

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="https://api.avalai.ir/v1", api_key=AVALAI_API_KEY)

response = client.images.generate(
    model="dall-e-3",
    prompt="A clean futuristic illustration of a student learning artificial intelligence and machine learning",
    size="1024x1024",
    quality="standard",
    n=1,
)

image_url = response.data[0].url

In [ ]:
image_url

In [ ]:
from openai import OpenAI # pip install -U openai

client = OpenAI(
    base_url="https://api.avalai.ir/v1",
    api_key=AVALAI_API_KEY
)

audio_file = open("sample_audio.m4a", "rb")  # Provide or upload an audio file
transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file,
    response_format="text"
)
print(transcription)

In [ ]:
from langchain_openai import ChatOpenAI
# pip install -U langchain_openai
import base64
import os

base_url = "https://api.avalai.ir/v1"
api_key=AVALAI_API_KEY

model_name = "gpt-4o"

# model_name = "gemini-2.0-flash-exp"

# model_name = "anthropic.claude-3-5-sonnet-20240620-v1:0"

llm = ChatOpenAI(
    base_url=base_url,
    model=model_name,
    api_key=api_key,
)

IMAGE_PATH = "sample.png"  # Provide or upload an image with this filename

# Open the image file and encode it as a base64 string
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

base64_image = encode_image(IMAGE_PATH)
image_ext = os.path.splitext(IMAGE_PATH)[1][1:]

messages = [
    {
        "role": "system",
        "content": "You are a helpful Persian assistant.",
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Tell me about this photo what is it, describe it in persian.",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/{image_ext};base64,{base64_image}",
                    "detail": "auto",
                },
            },
        ],
    },
]

ai_message = llm.invoke(messages)
print(ai_message.content)

# Part 4 — OpenRouter & Multimodal Models

In [ ]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)

completion = client.chat.completions.create(
  extra_headers={
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  model="google/gemma-3-12b-it:free",
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "در مورد این تصویر در نقش یک معلم نقاشی برای کودکان توضیح بده. توضیح در ۲۰ کلمه خلاصه شود"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://news-cdn.varzesh3.com/pictures/2025/08/13/D/15r3kcl17.webp?w=800"
          }
        }
      ]
    }
  ]
)
print(completion.choices[0].message.content)

#Let's use cURL in terminal

In [ ]:
curl https://openrouter.ai/api/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENROUTER_API_KEY" \
  -d '{
  "model": "openai/gpt-oss-20b:free",
  "messages": [
    {
      "role": "user",
      "content": "What is the meaning of life?"
    }
  ]
}'

# Part 5 — Interactive Gradio Applications

In [ ]:
!pip install -q gradio

In [ ]:
!pip install -q openai

In [ ]:
import gradio as gr
from openai import OpenAI  # Make sure openai is installed: pip install -U openai

# Initialize the OpenAI client with AvalAI API
client = OpenAI(
    base_url="https://api.avalai.ir/v1",
    api_key=AVALAI_API_KEY
)

# Function to generate speech and return audio file
def text_to_speech(text):
    speech_file_path = "generated_speech.mp3"  # Output file

    try:
        # Generate speech using AvalAI API
        response = client.audio.speech.create(
            model="tts-1",
            voice="alloy",
            input=text,
        )

        # Save the generated speech to a file
        response.stream_to_file(speech_file_path)

        return speech_file_path  # Returning the file path for playback & download

    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# Gradio UI
iface = gr.Interface(
    fn=text_to_speech,
    inputs=gr.Textbox(label="Enter Text"),
    outputs=gr.Audio(label="Generated Speech"),
    title="🗣️ AI-Powered Text-to-Speech (TTS)",
    description="Enter any text and get an AI-generated voice output using AvalAI.",
)

# Launch the web app
iface.launch()

In [ ]:
!pip install -U openai
!pip install -U langchain
!pip install -U langchain_openai

In [ ]:
import gradio as gr
import base64
import os
from io import BytesIO
from langchain_openai import ChatOpenAI

# API Configuration
base_url = "https://api.avalai.ir/v1"
api_key=AVALAI_API_KEY
model_name = "gpt-4o"
# Change model if needed: "gemini-2.0-flash-exp" or "anthropic.claude-3-5-sonnet-20240620-v1:0"

# Initialize LangChain OpenAI Model
llm = ChatOpenAI(
    base_url=base_url,
    model=model_name,
    api_key=api_key,
)

# Function to Convert Image to Base64
def encode_image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")  # Save as PNG for consistency
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# Function to Process Image and Get AI Description in Persian
def analyze_image(image):
    try:
        # Convert Image to Base64
        image_base64 = encode_image_to_base64(image)
        image_ext = "png"  # Default format for uploaded images

        # Messages for AI
        messages = [
            {"role": "system", "content": "You are a helpful Persian assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Describe the image in Farsi."},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/{image_ext};base64,{image_base64}",
                            "detail": "auto",
                        },
                    },
                ],
            },
        ]

        # Get AI-generated response
        ai_message = llm.invoke(messages)

        return ai_message.content  # Return AI description in Persian

    except Exception as e:
        return f"⚠️ خطا: {str(e)}"

# Gradio Web Interface
iface = gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="pil"),  # Allow users to upload an image
    outputs=gr.Textbox(label="توضیح تصویر توسط هوش مصنوعی"),
    title="🖼️ تحلیل تصویر با هوش مصنوعی (به زبان فارسی)",
    description="تصویری را بارگذاری کنید تا هوش مصنوعی توضیحی در مورد آن به زبان فارسی ارائه دهد.",
)

# Launch Web App
iface.launch()

In [ ]:
import gradio as gr
from langchain_openai import ChatOpenAI

# API Configuration
base_url = "https://api.avalai.ir/v1"
api_key=AVALAI_API_KEY

# Default model (Change if needed)
model_name = "gpt-4o"

# Initialize LangChain OpenAI Model
def get_llm(model_name):
    return ChatOpenAI(
        model=model_name,
        base_url=base_url,
        api_key=api_key,
    )

# Function to handle user input
def chat_with_ai(user_input, history, model_name):
    llm = get_llm(model_name)  # Get LLM instance for selected model

    messages = [{"role": "system", "content": "یک جوک خیلی بامزه و خلاقانه به زبان فارسی بنویس شرایط کمتر از 200 کلمه باشد موضوع جدید و متفاوت باشد طوری که تکراری و کلیشه‌ای نباشد محتوای سیاسی یا توهین‌آمیز نداشته باشد مناسب باشد که بتوانم برای همه تعریف کنم حتی در جمع خانوادگی شوخی باید هوشمندانه و غافلگیرکننده باشد شخصیت‌ها و موقعیت داستانی را طوری بساز که در ذهن شنونده تصویرسازی شود لحن شیرین پرانرژی و خنده‌دار باشد"}]

    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # Add current user input
    messages.append({"role": "user", "content": user_input})

    # Get AI response
    ai_message = llm.invoke(messages)

    return ai_message.content  # Return AI response

# Gradio Chat Interface
chatbot = gr.ChatInterface(
    fn=lambda message, history: chat_with_ai(message, history, model_name),
    title="💬 AI Chatbot with GPT",
    description="Chat with AI models like GPT using API.",
)

# Launch the Web App
chatbot.launch()

## Notes

- API-dependent cells require valid credentials and may incur provider costs.
- Some examples rely on local files such as `sample.png` or `sample_audio.m4a`.
- The notebook intentionally keeps the examples simple and independent so each API or modality can be tested separately.